# Initialization, Gradient Flow, and Deep Network Trainability

## Learning Objectives

A neural network may be expressive enough to represent a function and still be
difficult to train.

As networks become deeper, the backward pass has to propagate gradients through
many layers.

This raises several questions:

- how do gradients move through a deep network?
- why can they become extremely small or extremely large?
- how does parameter initialization affect activations and gradients?
- why do activation functions matter for gradient flow?
- what problem do Xavier and He initialization solve?

In this notebook we will deliberately build poorly initialized deep networks,
inspect their activations and gradients layer by layer, and then improve them.

The goal is not to memorize initialization formulas.

The goal is to understand **why those formulas exist**.

The central distinction for this notebook is:

```text
expressivity
→ can the network represent the function?

trainability
→ can optimization actually find useful parameters?
```

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch:", torch.__version__)

PyTorch: 2.14.1


## A Deeper Network

In the previous notebook we used only one hidden layer.

Now we will build a deeper network:

```text
input
↓
Linear
↓
ReLU
↓
Linear
↓
ReLU
↓
Linear
↓
ReLU
↓
Linear
↓
output
```

A deeper network repeatedly transforms its internal representation.

During the forward pass:

```text
activations move forward
```

During the backward pass:

```text
gradients move backward
```

The deeper the network becomes, the more transformations both signals must pass
through.

In [2]:
# A batch of 1,000 examples with 32 input features
batch_size = 1000
input_size = 32
hidden_size = 64

x = torch.randn(batch_size, input_size)

print("Input shape:", x.shape)
print("Input mean:", x.mean().item())
print("Input std:", x.std().item())

Input shape: torch.Size([1000, 32])
Input mean: 0.0027107344940304756
Input std: 1.0042102336883545


In [3]:
model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 1)
)

print(model)

Sequential(
  (0): Linear(in_features=32, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=64, bias=True)
  (5): ReLU()
  (6): Linear(in_features=64, out_features=64, bias=True)
  (7): ReLU()
  (8): Linear(in_features=64, out_features=1, bias=True)
)


In [4]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Total parameters:", total_params)

Total parameters: 14657


## Why Activation Scale Matters

Suppose the activations become progressively smaller:

```text
1.0
→ 0.3
→ 0.08
→ 0.01
→ 0.001
```

Then deeper layers may receive almost no meaningful signal.

Alternatively, suppose they grow:

```text
1.0
→ 3
→ 15
→ 100
→ 10,000
```

Then the network may become numerically unstable.

Ideally, initialization should keep activation scales reasonably controlled as
signals travel through the network.

We can diagnose this by measuring simple statistics such as:

```text
mean
standard deviation
```

at each layer.

In [5]:
activations = x

print(
    f"{'Input':20s} "
    f"mean={activations.mean().item(): .4f} "
    f"std={activations.std().item(): .4f}"
)

for i, layer in enumerate(model):
    activations = layer(activations)

    print(
        f"Layer {i:2d} "
        f"{layer.__class__.__name__:12s} "
        f"shape={tuple(activations.shape)} "
        f"mean={activations.mean().item(): .4f} "
        f"std={activations.std().item(): .4f}"
    )

Input                mean= 0.0027 std= 1.0042
Layer  0 Linear       shape=(1000, 64) mean=-0.0160 std= 0.5909
Layer  1 ReLU         shape=(1000, 64) mean= 0.2271 std= 0.3389
Layer  2 Linear       shape=(1000, 64) mean= 0.0106 std= 0.2483
Layer  3 ReLU         shape=(1000, 64) mean= 0.1040 std= 0.1468
Layer  4 Linear       shape=(1000, 64) mean= 0.0016 std= 0.1275
Layer  5 ReLU         shape=(1000, 64) mean= 0.0517 std= 0.0744
Layer  6 Linear       shape=(1000, 64) mean=-0.0123 std= 0.0885
Layer  7 ReLU         shape=(1000, 64) mean= 0.0303 std= 0.0494
Layer  8 Linear       shape=(1000, 1) mean= 0.0591 std= 0.0106


## Activation Statistics — Interpretation

The input was approximately standardized:

```text
mean ≈ 0
std  ≈ 1
```

As the signal moved through the network, its standard deviation generally
decreased:

```text
Input       std ≈ 1.004
Linear 1    std ≈ 0.591
ReLU 1      std ≈ 0.339

Linear 2    std ≈ 0.248
ReLU 2      std ≈ 0.147

Linear 3    std ≈ 0.128
ReLU 3      std ≈ 0.074

Linear 4    std ≈ 0.089
ReLU 4      std ≈ 0.049

Output      std ≈ 0.011
```

Two effects are visible.

### Linear layers change signal scale

Each linear layer computes:

$$
z = xW^T + b
$$

The scale of the resulting activations depends strongly on the scale of the
incoming values and the initialized weights.

### ReLU changes the activation distribution

ReLU computes:

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

Negative values are replaced with zero.

That changes both the mean and the variance of the distribution.

For example, after the first linear layer:

```text
mean ≈ -0.016
std  ≈ 0.591
```

and after ReLU:

```text
mean ≈ 0.227
std  ≈ 0.339
```

The mean becomes positive because negative activations are clipped to zero.

### Why this matters

By the final layer, the output standard deviation is only approximately:

```text
0.011
```

compared with approximately:

```text
1.004
```

at the input.

This does not by itself prove that the network cannot train.

However, it shows why initialization matters:

> Each layer transforms not only the information in the signal, but also its
> numerical scale.

In sufficiently deep networks, repeatedly shrinking or expanding signals can
make optimization difficult.

Next we will deliberately initialize the weights to extremely small values and
observe a much more severe version of this effect.

## Experiment: Deliberately Tiny Initialization

PyTorch initialized the previous network using sensible defaults.

To understand why initialization matters, we will override those parameters and
make every linear weight extremely small.

Suppose a layer repeatedly multiplies its input by very small weights.

Conceptually:

```text
signal
× small weights
↓
smaller signal
× small weights
↓
even smaller signal
× small weights
↓
almost zero
```

With enough layers, useful variation can disappear.

We will initialize every linear weight from a normal distribution with:

```text
standard deviation = 0.01
```

and set the biases to zero.

Then we will inspect the activation statistics again.

In [6]:
torch.manual_seed(42)

tiny_model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 1)
)

# Deliberately bad initialization
with torch.no_grad():
    for layer in tiny_model:
        if isinstance(layer, nn.Linear):
            layer.weight.normal_(mean=0.0, std=0.01)
            layer.bias.zero_()

In [7]:
activations = x

print(
    f"{'Input':20s} "
    f"mean={activations.mean().item(): .8f} "
    f"std={activations.std().item(): .8f}"
)

for i, layer in enumerate(tiny_model):
    activations = layer(activations)

    print(
        f"Layer {i:2d} "
        f"{layer.__class__.__name__:12s} "
        f"shape={tuple(activations.shape)} "
        f"mean={activations.mean().item(): .8f} "
        f"std={activations.std().item(): .8f}"
    )

Input                mean= 0.00271073 std= 1.00421023
Layer  0 Linear       shape=(1000, 64) mean=-0.00027898 std= 0.05754245
Layer  1 ReLU         shape=(1000, 64) mean= 0.02263815 std= 0.03363543
Layer  2 Linear       shape=(1000, 64) mean=-0.00001036 std= 0.00319425
Layer  3 ReLU         shape=(1000, 64) mean= 0.00125376 std= 0.00187942
Layer  4 Linear       shape=(1000, 64) mean=-0.00002356 std= 0.00018982
Layer  5 ReLU         shape=(1000, 64) mean= 0.00006283 std= 0.00010434
Layer  6 Linear       shape=(1000, 64) mean= 0.00000027 std= 0.00000935
Layer  7 ReLU         shape=(1000, 64) mean= 0.00000377 std= 0.00000568
Layer  8 Linear       shape=(1000, 1) mean= 0.00000051 std= 0.00000028


## Tiny Initialization — Interpretation

With weights initialized from:

```text
Normal(mean=0, std=0.01)
```

the forward signal collapsed rapidly through the network.

The activation standard deviation changed approximately as:

```text
Input       1.004
Linear 1    0.0575
ReLU 1      0.0336

Linear 2    0.00319
ReLU 2      0.00188

Linear 3    0.000190
ReLU 3      0.000104

Linear 4    0.00000935
ReLU 4      0.00000568

Output      0.00000028
```

Each layer multiplied its input by very small weights.

Because this happened repeatedly, the signal scale shrank by orders of magnitude.

Conceptually:

```text
meaningful input variation
↓
small weights
↓
smaller activations
↓
small weights again
↓
even smaller activations
↓
...
↓
almost no variation remains
```

By the final layer, many different inputs produce nearly the same output.

This is a forward-pass manifestation of poor initialization.

The next question is even more important:

> What happens to the gradients during backpropagation?

In [8]:
# Synthetic regression target
target = torch.randn(batch_size, 1)

# Forward pass
prediction = tiny_model(x)

loss = ((prediction - target) ** 2).mean()

print("Loss:", loss.item())

# Backward pass
tiny_model.zero_grad()
loss.backward()

Loss: 1.0088934898376465


In [9]:
print(
    f"{'Layer':20s} "
    f"{'weight grad mean':>18s} "
    f"{'weight grad std':>18s} "
    f"{'weight grad max':>18s}"
)

for i, layer in enumerate(tiny_model):
    if isinstance(layer, nn.Linear):
        grad = layer.weight.grad

        print(
            f"Linear layer {i:<7d} "
            f"{grad.abs().mean().item():18.10f} "
            f"{grad.std().item():18.10f} "
            f"{grad.abs().max().item():18.10f}"
        )

Layer                  weight grad mean    weight grad std    weight grad max
Linear layer 0             0.0000000612       0.0000000787       0.0000003296
Linear layer 2             0.0000000419       0.0000000583       0.0000003875
Linear layer 4             0.0000000365       0.0000000603       0.0000004090
Linear layer 6             0.0000000337       0.0000000730       0.0000011636
Linear layer 8             0.0000003615       0.0000003954       0.0000016384


## Tiny Initialization — Gradient Flow

With tiny weights, the forward activations collapsed toward zero.

The backward pass shows a related problem.

Mean absolute weight gradients were approximately:

```text
Layer 0  → 6.12e-8
Layer 2  → 4.19e-8
Layer 4  → 3.65e-8
Layer 6  → 3.37e-8
Layer 8  → 3.62e-7
```

The gradients in the early layers are extremely small.

This means those parameters would update only very slowly.

### Why?

Backpropagation repeatedly applies the chain rule.

For a simplified deep network:

```text
x
→ layer 1
→ layer 2
→ layer 3
→ loss
```

the gradient for an early parameter contains a product of derivatives from all
later operations.

Conceptually:

```text
early-layer gradient
=
later gradient
× derivative
× weight
× derivative
× weight
× ...
```

If many of those multiplicative terms are small, their product becomes very
small.

This is the core mechanism behind **vanishing gradients**.

The important idea is:

> Vanishing gradients are not caused by one small number. They emerge from
> repeatedly multiplying small factors through depth.

## Experiment: Deliberately Large Initialization

Tiny weights caused signals and gradients to shrink toward zero.

Now we will test the opposite failure mode.

If every layer uses weights that are too large, then repeated multiplication can cause
activations to grow rapidly through depth.

Conceptually:

```text
signal
× large weights
↓
larger signal
× large weights
↓
much larger signal
× large weights
↓
possibly unstable values
```

The same issue can affect backpropagation.

Because gradients are also propagated through repeated multiplication, large
factors can cause gradient magnitudes to grow dramatically.

This is the basic mechanism behind **exploding activations** and
**exploding gradients**.

We will initialize every linear weight from:

```text
Normal(mean=0, std=1.0)
```

which is intentionally much larger than a sensible initialization for these
layer sizes.

In [10]:
torch.manual_seed(42)

large_model = nn.Sequential(
    nn.Linear(32, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 1)
)

with torch.no_grad():
    for layer in large_model:
        if isinstance(layer, nn.Linear):
            layer.weight.normal_(mean=0.0, std=1.0)
            layer.bias.zero_()

In [11]:
activations = x

print(
    f"{'Input':20s} "
    f"mean={activations.mean().item(): .6f} "
    f"std={activations.std().item(): .6f}"
)

for i, layer in enumerate(large_model):
    activations = layer(activations)

    print(
        f"Layer {i:2d} "
        f"{layer.__class__.__name__:12s} "
        f"shape={tuple(activations.shape)} "
        f"mean={activations.mean().item(): .6f} "
        f"std={activations.std().item(): .6f}"
    )

Input                mean= 0.002711 std= 1.004210
Layer  0 Linear       shape=(1000, 64) mean=-0.027898 std= 5.754245
Layer  1 ReLU         shape=(1000, 64) mean= 2.263815 std= 3.363543
Layer  2 Linear       shape=(1000, 64) mean=-0.103582 std= 31.942480
Layer  3 ReLU         shape=(1000, 64) mean= 12.537594 std= 18.794182
Layer  4 Linear       shape=(1000, 64) mean=-23.561449 std= 189.824005
Layer  5 ReLU         shape=(1000, 64) mean= 62.832367 std= 104.343430
Layer  6 Linear       shape=(1000, 64) mean= 26.653410 std= 934.765015
Layer  7 ReLU         shape=(1000, 64) mean= 376.537506 std= 567.764343
Layer  8 Linear       shape=(1000, 1) mean= 5058.308594 std= 2786.624023


In [12]:
prediction = large_model(x)

loss = ((prediction - target) ** 2).mean()

print("Loss:", loss.item())

large_model.zero_grad()
loss.backward()

print(
    f"{'Layer':20s} "
    f"{'weight grad mean':>18s} "
    f"{'weight grad std':>18s} "
    f"{'weight grad max':>18s}"
)

for i, layer in enumerate(large_model):
    if isinstance(layer, nn.Linear):
        grad = layer.weight.grad

        print(
            f"Linear layer {i:<7d} "
            f"{grad.abs().mean().item():18.6f} "
            f"{grad.std().item():18.6f} "
            f"{grad.abs().max().item():18.6f}"
        )

Loss: 33344548.0
Layer                  weight grad mean    weight grad std    weight grad max
Linear layer 0             80740.468750      108395.968750      516280.406250
Linear layer 2            190100.093750      262654.593750     1503612.750000
Linear layer 4            226650.250000      425091.625000     3730300.000000
Linear layer 6            313598.500000      704008.750000    10359666.000000
Linear layer 8           4076958.750000     4404898.000000    18176802.000000


## Large Initialization — Interpretation

With weights initialized from:

```text
Normal(mean=0, std=1.0)
```

the network became numerically unstable.

The loss increased to approximately:

```text
33,344,548
```

and the gradient magnitudes became extremely large.

Mean absolute weight gradients were approximately:

```text
Layer 0  → 8.07e4
Layer 2  → 1.90e5
Layer 4  → 2.27e5
Layer 6  → 3.14e5
Layer 8  → 4.08e6
```

The largest individual gradient reached approximately:

```text
1.82e7
```

This is the opposite failure mode from tiny initialization.

With very small weights:

```text
activations shrink
gradients shrink
```

With very large weights:

```text
activations grow
gradients grow
```

The underlying mechanism is again repeated multiplication through depth.

During the forward pass:

```text
activation
× weight
× weight
× weight
...
```

can cause signal magnitudes to grow rapidly.

During the backward pass, the chain rule creates another product of derivatives
and weights.

If many of those factors are large, gradients can grow rapidly as well.

This is the basic mechanism behind **exploding gradients**.

### The initialization problem

A useful initialization should avoid both extremes:

```text
too small
→ vanishing activations and gradients

too large
→ exploding activations and gradients
```

The goal is to preserve a reasonable signal scale as information moves forward
and gradients move backward through the network.